# NAS7 데이터 처리 — 서울시 건설알림이 공사 데이터

## 데이터 개요
- **파일**: `서울시건설알림이정보.csv`
- **내용**: 서울시 등록 공사 프로젝트 목록
- **연구 활용 목적**: 공사 현장 근처 도로에서 재비산먼지가 높아지는 현상 포착

## 주요 컬럼
| 컬럼 | 설명 |
|------|------|
| 사업착수일(계약일) | 공사 시작일 (YYYYMMDD) |
| 사업기간 | `YYYYMMDD~YYYYMMDD` 형식 |
| 프로젝트 종료여부 | 0=진행중, 1=완료, 2=예정, 3=중지 |
| 위치좌표(위도/경도) | 공사 위치 |
| 사업금액(억원) | 공사 규모 프록시 |

## 처리 결과
- **처리 스크립트**: `RoadExtension_V3/step_construction.py`
- **출력**: `RoadExtension_V3/checkpoints/construction_cache.csv`
  - 컬럼: CELL_ID, date, n_active_const, total_amount_억, max_amount_억, is_large_const

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import sqlite3

construction_path = "/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/서울시건설알림이정보.csv"
df = pd.read_csv(construction_path, encoding='cp949')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(3)

In [ ]:
# 날짜 파싱
df['start_date'] = pd.to_datetime(df['사업착수일(계약일)'].astype(str), format='%Y%m%d', errors='coerce')
df['end_date'] = pd.to_datetime(
    df['사업기간'].astype(str).str.split('~').str[-1].str.strip(),
    format='%Y%m%d', errors='coerce')
df['status'] = df['프로젝트 종료여부(진행:0 종료:1,예정:2, 중지:3)']
df['amount'] = df['사업금액(억원)']

# 연구 기간(2023~2025) 관련 필터
study_start = pd.Timestamp('2023-01-01')
study_end   = pd.Timestamp('2025-12-31')

relevant = df[
    (df['start_date'] <= study_end) &
    (df['end_date']   >= study_start) &
    (df['status'] != 3) &
    df['위치좌표(위도)'].between(37.4, 37.7) &
    df['위치좌표(경도)'].between(126.7, 127.2)
].copy()

print(f"연구 기간 관련 공사: {len(relevant)}건")
print("상태:", relevant['status'].value_counts().to_dict())
print("사업금액 분포:")
print(relevant['amount'].describe())

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 공사 기간 분포
relevant['duration'] = (relevant['end_date'] - relevant['start_date']).dt.days
axes[0].hist(relevant['duration'].clip(0, 1000), bins=50, color='#2196F3', edgecolor='none', alpha=0.8)
axes[0].set_title('Construction Duration (days)')
axes[0].set_xlabel('days')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# 사업금액 분포 (log)
axes[1].hist(np.log1p(relevant['amount']), bins=50, color='#FF9800', edgecolor='none', alpha=0.8)
axes[1].set_title('Construction Amount (log1p 억원)')
axes[1].set_xlabel('log1p(억원)')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

# 연도별 착수 건수
year_cnt = relevant['start_date'].dt.year.value_counts().sort_index()
axes[2].bar(year_cnt.index.astype(str), year_cnt.values, color='#4CAF50', alpha=0.8)
axes[2].set_title('Projects by Start Year')
axes[2].set_xlabel('year')
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('/tmp/construction_eda.png', dpi=120, bbox_inches='tight')
plt.close()
print('EDA 그래프 저장 완료')

In [ ]:
# 도로 PM과의 상관관계 예측 — 날짜별 활성 공사 수 vs 도로 PM
import openpyxl

# 일별 활성 공사 수 (전서울)
dates = pd.date_range('2023-01-01', '2025-12-31', freq='D')
daily_const = []
for d in dates:
    n = ((relevant['start_date'] <= d) & (relevant['end_date'] >= d)).sum()
    daily_const.append({'date': d, 'n_active': n})
daily_df = pd.DataFrame(daily_const)

# 도로 PM 일별 평균
dfs = []
for yr in ['23','24','25']:
    wb = openpyxl.load_workbook(
        f'/home/data/youngwoong/ST-GNN_Dataset/0-2. 도로 미세먼지/서울 비재산먼지_{yr}년도.xlsx',
        data_only=True)
    rows = list(wb.active.iter_rows(values_only=True))
    wb.close()
    dfs.append(pd.DataFrame(rows[1:], columns=rows[0]))
road_df = pd.concat(dfs, ignore_index=True)
road_df['재비산먼지'] = pd.to_numeric(road_df['재비산먼지 평균농도(㎍/㎥)'], errors='coerce')
road_df['date'] = pd.to_datetime(road_df['측정일자']).dt.normalize()
road_daily = road_df.dropna(subset=['재비산먼지']).groupby('date')['재비산먼지'].mean().reset_index()

merged = road_daily.merge(daily_df, on='date', how='inner')
corr = merged[['재비산먼지','n_active']].corr().iloc[0,1]
print(f"전서울 활성 공사 수 vs 도로PM 상관계수: r={corr:.3f}")
print(f"활성 공사 수 범위: {daily_df['n_active'].min()}~{daily_df['n_active'].max()}, 평균={daily_df['n_active'].mean():.0f}")

In [ ]:
# 실제 처리 스크립트 실행
import subprocess
result = subprocess.run(
    ['python3', '/workspace/ST-GNN Modeling/RoadExtension_V3/step_construction.py'],
    capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[:500])

In [ ]:
# 결과 확인
cache = pd.read_csv('/workspace/ST-GNN Modeling/RoadExtension_V3/checkpoints/construction_cache.csv',
                    parse_dates=['date'])
print("Shape:", cache.shape)
print(cache.describe())
print("\n공사 없는 날 비율: {:.1f}%".format(100*(cache['n_active_const']==0).mean()))
print("공사 1개 이상 날 비율: {:.1f}%".format(100*(cache['n_active_const']>0).mean()))